# ITS — Plate detection + OCR training (Colab GPU)

Thin driver. All real logic lives in `tools/train_plates.py` in the repo, so
you edit it in **VS Code**, push, and re-run the cells here — no code is
trapped in this notebook.

**Workflow:** edit in VS Code -> `git push` -> re-run cell 2 (`git pull`) -> train.

Before spending GPU time, read the resolution note in step 6: OCR needs ~100px
plate width and `samples/street_egypt.mp4` provides ~34px. Training cannot fix
that. Detection and colour work fine there.

## 1. Check the GPU

Runtime -> Change runtime type -> **T4 GPU**. If this prints `cpu`, stop and fix it.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print('NO GPU — set Runtime > Change runtime type > T4 GPU')

## 2. Pull the repo

Clones on the first run, pulls on every run after. Re-run this cell after each
`git push` from VS Code.

The repo is public, so no token is needed to read. If you make it private,
use a fine-grained PAT with *Contents: read* and keep it in Colab **Secrets**
(the key icon), never pasted into a cell.

In [ ]:
import os, subprocess
REPO = 'https://github.com/yousseffbassemm/ITS-project-Elsewedy.git'
DIR = '/content/its-traffic'

if os.path.isdir(DIR + '/.git'):
    print(subprocess.run(['git','-C',DIR,'pull','--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    subprocess.run(['git','clone',REPO,DIR], check=True)

os.chdir(DIR)
print('branch:', subprocess.run(['git','branch','--show-current'],
                               capture_output=True, text=True).stdout.strip())
print('head:  ', subprocess.run(['git','log','--oneline','-1'],
                               capture_output=True, text=True).stdout.strip())

Switch to the plates branch if it has not been merged yet:

In [ ]:
!git checkout plates-anpr && git pull --ff-only

## 3. Dependencies

Colab already ships torch built for its GPU — do **not** reinstall torch, it
will pull a CPU build or a mismatched CUDA and silently drop you to CPU.

In [ ]:
!pip install -q ultralytics supervision
import ultralytics; ultralytics.checks()

## 4. Get the datasets

Not vendored in the repo — large and separately licensed. Two stages need two
datasets (sources and licences in `docs/anpr-plan.md`):

| stage | needs | suggested source |
|---|---|---|
| detect | vehicle images, box around the plate | Roboflow *Egypt Car Plate*, or Kaggle license-plate sets |
| ocr | plate crops, box per character | **EALPR** (Egyptian, Arabic, 27 classes) |

Put your Roboflow key in Colab **Secrets** as `ROBOFLOW_API_KEY` rather than
typing it into a cell — notebook output gets committed by accident.

In [ ]:
from google.colab import userdata
import os

try:
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
    print('Roboflow key loaded from Secrets')
except Exception as e:
    print('No ROBOFLOW_API_KEY secret set —', e)
    print('Add it via the key icon in the left sidebar, or download data manually.')

In [ ]:
# Example: Roboflow. Replace workspace/project/version with the dataset you chose.
!pip install -q roboflow
from roboflow import Roboflow
import os

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
ds = (rf.workspace('yousef-gamal').project('egypt-car-plate')
        .version(4).download('yolov8', location='data/plates/detect'))
print('downloaded to', ds.location)

In [ ]:
# Verify what actually landed, BEFORE training on it.
import yaml, pathlib, collections

for stage in ('detect', 'ocr'):
    p = pathlib.Path(f'data/plates/{stage}/data.yaml')
    if not p.exists():
        print(f'{stage}: MISSING {p}')
        continue
    cfg = yaml.safe_load(p.read_text())
    root = p.parent
    counts = {s: len(list((root/s/'images').glob('*'))) for s in ('train','valid','test')
              if (root/s/'images').is_dir()}
    print(f'{stage}: {len(cfg.get("names", []))} classes, images {counts}')
    print('   names:', cfg.get('names'))

## 5. Train

Both stages call the repo script, so the augmentation choices are versioned and
reviewable. Two of them matter enough to repeat here:

- **`fliplr=0`** — mirroring a plate reverses reading order and maps some Arabic
  glyphs onto each other. This is the augmentation most likely to quietly ruin
  the OCR model.
- **`mosaic=0` for OCR** — stitching four crops invents plates that do not exist.

In [ ]:
!python -m tools.train_plates --stage detect --epochs 80 --batch 16 --device 0

In [ ]:
!python -m tools.train_plates --stage ocr --epochs 120 --batch 32 --device 0

## 6. Get the target clip into Colab

`samples/*.mp4` is **gitignored** (large, and it is real CCTV of identifiable
vehicles), so the clone does not include it. The reality check in the next step
needs it. Upload it once, or keep it on Drive.

Treat it as personal data: do not leave it in a shared Colab, and clear the
outputs before sharing this notebook.

In [ ]:
import pathlib, os

clip = pathlib.Path('samples/street_egypt.mp4')
clip.parent.mkdir(exist_ok=True)

if clip.exists():
    print('already present:', clip, f'{clip.stat().st_size/1e6:.1f} MB')
else:
    drive_copy = '/content/drive/MyDrive/its-plates/street_egypt.mp4'
    if os.path.exists(drive_copy):
        import shutil; shutil.copy(drive_copy, clip)
        print('copied from Drive')
    else:
        from google.colab import files
        up = files.upload()          # pick street_egypt.mp4
        for n in up:
            os.replace(n, clip)
        print('uploaded ->', clip)

## 7. The reality check

Run the trained detector against the **actual target footage**. A model can
score well on its training domain and find nothing on your camera; this is where
that shows up, rather than in front of the mentor.

Expect `ocr_viable: False` on `street_egypt.mp4`. That is the correct answer for
that clip, not a training failure — the plates are ~34px wide and OCR needs ~100.

In [ ]:
!python -m tools.plate_footage_check --videos samples/street_egypt.mp4 --geometric-only

In [ ]:
import json, pathlib
p = pathlib.Path('models/train_report.json')
print(json.dumps(json.loads(p.read_text()), indent=2) if p.exists() else 'no report yet')

## 8. Get the weights out

Colab wipes the VM when it disconnects. Download the weights **before** you
close the tab. They are ~20MB each.

Do not commit `.pt` files — `.gitignore` already excludes them.

In [ ]:
from google.colab import files
import pathlib

for name in ('plate_detect.pt', 'plate_ocr.pt'):
    p = pathlib.Path('models')/name
    if p.exists():
        print(f'{name}: {p.stat().st_size/1e6:.1f} MB')
        files.download(str(p))
    else:
        print(f'{name}: not produced')

Optional — keep a copy on Drive so a disconnect is not fatal:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/its-plates && cp models/*.pt /content/drive/MyDrive/its-plates/ 2>/dev/null; ls -la /content/drive/MyDrive/its-plates/

## 9. Use them locally

Drop both files into `models/` in your local repo, then:

```powershell
$env:ITS_PLATE_MODEL="models/plate_detect.pt"
$env:ITS_PLATE_OCR_MODEL="models/plate_ocr.pt"
```

A note on the data: plate crops are **personal data**. `.gitignore` already
blocks `data/` for this reason. Keep training images off public repos and out of
notebook outputs.